# RQ4 ONLY — HCLA Component Ablation

Standalone notebook for independent execution.

- RQ4 only
- 50 cases × 4 conditions × 3 reps
- Supports resuming from checkpoints
- Records execution time
- Saves results as CSV / PKL
- Includes Friedman tests, pairwise Wilcoxon tests, and Holm correction

**Note:** Shared functions and variables such as `load_evidence_checkpoint()`, `qwen_generate()`, `judge_narrative()`, `judge_quality()`, and `RQ345_OUTPUT_DIR` must be defined in preceding shared cells or in an existing notebook.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# ============================================================
# RQ4 ONLY - HCLA Component Ablation
# Fully standalone execution / Colab + OpenRouter
# ============================================================

import os
import re
import json
import time
import pickle
import random
import threading
from pathlib import Path
from itertools import combinations
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import pandas as pd
from scipy.stats import friedmanchisquare, wilcoxon
from openai import OpenAI


# ============================================================
# 0. CONFIG
# ============================================================

RQ345_OUTPUT_DIR = (
    "/content/drive/MyDrive/Colab Notebooks/Results/"
    "HCLA_MultiModel_Qwen_Llama_SHAP"
)

RQ4_N_CASES = 50
RQ4_REPS = 3
RQ4_WORKERS = 2   # Reduce to 1 if HTTP 402 errors occur frequently. Failed jobs are not checkpointed and will be retried on the next run.

QWEN_MODEL = "qwen/qwen3-235b-a22b-2507"
LLAMA_JUDGE_MODEL = "meta-llama/llama-3.3-70b-instruct"

GEN_TEMPERATURE = 0.2
GEN_MAX_TOKENS = 450

JUDGE_TEMPERATURE = 0.0
GROUND_JUDGE_MAX_TOKENS = 900
QUALITY_JUDGE_MAX_TOKENS = 220

API_MAX_RETRIES = 2
API_402_WAIT_SEC = 5
API_RETRY_BASE_SEC = 5

RQ4_CONDITIONS = [
    "Full_HCLA",
    "Without_RoleSeparation",
    "Without_EvidenceConstraint",
    "Without_InteractiveRefinement",
]

RQ4_OUTPUT_DIR = os.path.join(
    RQ345_OUTPUT_DIR,
    "RQ4_Component_Ablation"
)

CHECKPOINT_DIR = os.path.join(
    RQ345_OUTPUT_DIR,
    "_checkpoints"
)

os.makedirs(RQ4_OUTPUT_DIR, exist_ok=True)

RQ4_CHECKPOINT = os.path.join(
    RQ4_OUTPUT_DIR,
    "rq4_checkpoint.csv"
)

RQ4_FULL_PATH = os.path.join(
    RQ4_OUTPUT_DIR,
    "rq4_ablation_full.csv"
)

RQ4_SUMMARY_PATH = os.path.join(
    RQ4_OUTPUT_DIR,
    "rq4_ablation_summary.csv"
)

RQ4_RUNTIME_PATH = os.path.join(
    RQ4_OUTPUT_DIR,
    "rq4_runtime.csv"
)

RQ4_PKL_PATH = os.path.join(
    RQ4_OUTPUT_DIR,
    "rq4_results_qwen_llama.pkl"
)

RQ4_STATS_PATH = os.path.join(
    RQ4_OUTPUT_DIR,
    "rq4_statistical_tests.csv"
)


# ============================================================
# 1. OPENROUTER CLIENT
# ============================================================

def get_openrouter_api_key():
    # Configure OPENROUTER_API_KEY in your runtime environment or Colab Secrets.
    key = os.environ.get("OPENROUTER_API_KEY", "").strip()
    if not key:
        try:
            from google.colab import userdata
            key = (userdata.get("OPENROUTER_API_KEY") or "").strip()
        except (ImportError, KeyError, AttributeError):
            pass
    if not key:
        raise RuntimeError("Configure OPENROUTER_API_KEY in the environment or Colab Secrets.")
    return key


client = OpenAI(
    api_key=get_openrouter_api_key(),
    base_url="https://openrouter.ai/api/v1"
)

print("✅ OpenRouter client ready")
print("Generator:", QWEN_MODEL)
print("Judge    :", LLAMA_JUDGE_MODEL)


# ============================================================
# 2. COMMON API CALL
# ============================================================

def is_402_error(ex):
    status = getattr(ex, "status_code", None)

    if status == 402:
        return True

    msg = str(ex).lower()

    return (
        "error code: 402" in msg
        or "in_flight_budget_exhausted" in msg
        or "available credits" in msg
        or "openrouter_credits" in msg
    )


def call_model(
    model,
    prompt,
    temperature=0.0,
    max_tokens=600,
    max_retries=API_MAX_RETRIES
):
    last_ex = None

    for attempt in range(1, max_retries + 1):
        try:
            response = client.chat.completions.create(
                model=model,
                messages=[
                    {"role": "user", "content": prompt}
                ],
                temperature=temperature,
                max_tokens=max_tokens
            )

            text = response.choices[0].message.content

            if text is None or not str(text).strip():
                raise ValueError("Empty model response")

            return str(text).strip()

        except Exception as ex:
            last_ex = ex

            if attempt >= max_retries:
                break

            if is_402_error(ex):
                # For HTTP 402, retry briefly once rather than waiting for a long time.
                # If the second attempt fails, only the current job is marked as failed;
                # it is not checkpointed and will be retried automatically on the next run.
                wait = API_402_WAIT_SEC
                print(
                    f"⚠️ 402 short retry [{attempt}/{max_retries}] "
                    f"{type(ex).__name__}: {str(ex)[:220]} "
                    f"-> Retry once after {wait:.1f}s"
                )
            else:
                wait = min(
                    30,
                    API_RETRY_BASE_SEC * (2 ** (attempt - 1))
                )
                wait += random.uniform(0, 1.5)
                print(
                    f"⚠️ API retry [{attempt}/{max_retries}] "
                    f"{type(ex).__name__}: {str(ex)[:220]} "
                    f"-> {wait:.1f}s"
                )

            time.sleep(wait)

    raise RuntimeError(
        f"API failed after {max_retries} attempts: {last_ex}"
    )


def qwen_generate(
    prompt,
    temperature=GEN_TEMPERATURE
):
    return call_model(
        QWEN_MODEL,
        prompt,
        temperature=temperature,
        max_tokens=GEN_MAX_TOKENS
    )


# ============================================================
# 3. LOAD EVIDENCE CHECKPOINT
# ============================================================

def load_evidence_checkpoint():
    candidates = [
        os.path.join(
            CHECKPOINT_DIR,
            "evidence_packages.pkl"
        ),
        os.path.join(
            RQ345_OUTPUT_DIR,
            "evidence_packages.pkl"
        ),
    ]

    found = None

    for p in candidates:
        if os.path.exists(p):
            found = p
            break

    if found is None:
        matches = list(
            Path(RQ345_OUTPUT_DIR).rglob(
                "evidence_packages.pkl"
            )
        )

        if matches:
            found = str(matches[0])

    if found is None:
        raise FileNotFoundError(
            "Could not find evidence_packages.pkl.\n"
            + "\n".join(candidates)
        )

    with open(found, "rb") as f:
        evidence_packages = pickle.load(f)

    if isinstance(evidence_packages, dict):
        for key in [
            "evidence_packages",
            "evidence",
            "packages",
            "data"
        ]:
            if key in evidence_packages:
                evidence_packages = evidence_packages[key]
                break

    if not isinstance(
        evidence_packages,
        (list, tuple)
    ):
        raise TypeError(
            "Evidence checkpoint is not a list or tuple."
        )

    evidence_packages = list(
        evidence_packages
    )

    if len(evidence_packages) == 0:
        raise RuntimeError(
            "Evidence checkpoint is empty."
        )

    for i, e in enumerate(evidence_packages):
        if "case_id" not in e:
            e["case_id"] = str(i)

    print("✅ Evidence checkpoint:", found)
    print(
        "✅ Evidence packages loaded:",
        len(evidence_packages)
    )

    return evidence_packages


# ============================================================
# 4. RQ4 PROMPT DEFINITIONS
# ============================================================

FULL_INST = """
You are the HCLA Justification component.

Use ONLY the supplied evidence.
Do not introduce external facts.
Internally verify every factual statement against the evidence.
State limitations when the supplied evidence is insufficient.

Write a concise expert-style explanation.
""".strip()


ROLE_PROMPTS = {
    "Without_RoleSeparation": """
You are an AI assistant.
Directly explain the prediction in one pass.
Use the supplied evidence as context.
""".strip(),

    "Without_EvidenceConstraint": """
Explain the prediction using the supplied information
and any relevant reasoning you consider useful.
""".strip(),
}


# ============================================================
# 5. RQ4 GENERATION
# ============================================================

def rq4_generate(e, condition):
    evidence = json.dumps(
        e,
        ensure_ascii=False,
        indent=2,
        default=str
    )

    if condition == "Full_HCLA":
        draft_prompt = (
            FULL_INST
            + "\n\nEvidence:\n"
            + evidence
            + "\n\nWrite a 2-4 sentence draft."
        )

        draft = qwen_generate(
            draft_prompt
        )

        refine_prompt = (
            "You are the HCLA refinement component.\n"
            "Review the draft ONLY against the supplied evidence.\n"
            "Remove unsupported, speculative, or overstated claims.\n"
            "Preserve useful evidence-based reasoning.\n"
            "Return ONLY the revised paragraph.\n\n"
            "Evidence:\n"
            + evidence
            + "\n\nDraft:\n"
            + draft
        )

        return qwen_generate(
            refine_prompt,
            temperature=0.0
        )

    elif condition == "Without_InteractiveRefinement":
        prompt = (
            FULL_INST
            + "\n\nEvidence:\n"
            + evidence
            + "\n\nWrite a concise explanation."
        )

        return qwen_generate(
            prompt
        )

    elif condition in ROLE_PROMPTS:
        prompt = (
            ROLE_PROMPTS[condition]
            + "\n\nEvidence:\n"
            + evidence
            + "\n\nWrite a concise explanation."
        )

        return qwen_generate(
            prompt
        )

    else:
        raise ValueError(
            f"Unknown RQ4 condition: {condition}"
        )


# ============================================================
# 6. GROUNDEDNESS JUDGE
# No JSON used
# ============================================================

GROUND_JUDGE_INST = """
You are a strict evaluator of an AI explanation.

Evaluate EVERY factual claim in the explanation against ONLY the supplied evidence.

Allowed labels:
SUPPORTED = directly supported by the supplied evidence
UNSUPPORTED = not supported by the supplied evidence
CONTRADICTED = conflicts with the supplied evidence

Do not use outside knowledge.
Do not judge style.
Do not omit factual claims.

Return ONLY one claim per line using exactly:

SUPPORTED|claim text
UNSUPPORTED|claim text
CONTRADICTED|claim text

Do not add numbering, markdown, headings, or commentary.
""".strip()


def parse_groundedness(raw):
    pattern = re.compile(
        r"(?i)\b"
        r"(SUPPORTED|UNSUPPORTED|CONTRADICTED)"
        r"\s*\|\s*"
    )

    matches = list(
        pattern.finditer(raw)
    )

    claims = []

    for i, m in enumerate(matches):
        label = m.group(1).upper()
        start = m.end()

        end = (
            matches[i + 1].start()
            if i + 1 < len(matches)
            else len(raw)
        )

        claim = raw[
            start:end
        ].strip(
            " \n\r\t;|-"
        )

        if claim:
            claims.append({
                "label": label,
                "claim": claim
            })

    if not claims:
        raise ValueError(
            "Grounded judge parse failed: "
            + raw[:500]
        )

    return claims


def judge_narrative(
    e,
    txt,
    parse_retries=3
):
    evidence = json.dumps(
        e,
        ensure_ascii=False,
        indent=2,
        default=str
    )

    prompt = (
        GROUND_JUDGE_INST
        + "\n\nSUPPLIED EVIDENCE:\n"
        + evidence
        + "\n\nEXPLANATION:\n"
        + str(txt)
    )

    last_ex = None

    for attempt in range(
        1,
        parse_retries + 1
    ):
        try:
            raw = call_model(
                LLAMA_JUDGE_MODEL,
                prompt,
                temperature=JUDGE_TEMPERATURE,
                max_tokens=GROUND_JUDGE_MAX_TOKENS
            )

            claims = parse_groundedness(
                raw
            )

            return {
                "claims": claims,
                "raw": raw
            }

        except Exception as ex:
            last_ex = ex

            if attempt < parse_retries:
                print(
                    f"⚠️ Grounded judge retry "
                    f"[{attempt}/{parse_retries}] "
                    f"{type(ex).__name__}: "
                    f"{str(ex)[:220]}"
                )
                time.sleep(3)

    raise RuntimeError(
        f"Groundedness judge failed: {last_ex}"
    )


def rq4_grounded_rates(e, txt):
    judged = judge_narrative(
        e,
        txt
    )

    labels = [
        str(
            x.get(
                "label",
                ""
            )
        ).upper()
        for x in judged.get(
            "claims",
            []
        )
    ]

    n = len(labels)

    if n == 0:
        raise RuntimeError(
            "Groundedness judge returned zero claims."
        )

    return {
        "ESR": (
            labels.count(
                "SUPPORTED"
            ) / n
        ),
        "UCR": (
            labels.count(
                "UNSUPPORTED"
            ) / n
        ),
        "CR": (
            labels.count(
                "CONTRADICTED"
            ) / n
        ),
        "claim_count": n
    }


# ============================================================
# 7. QUALITY JUDGE
# No JSON used
# ============================================================

QUALITY_JUDGE_INST = """
You are a strict evaluator of an explanation for an anomaly detector.

Evaluate the explanation using ONLY the supplied evidence.

Score each dimension from 1 to 5.

CLARITY
1 = confusing or largely uninterpretable
2 = understandable with substantial effort
3 = generally understandable but has notable issues
4 = clear and coherent
5 = exceptionally clear and concise

TRUST_SUPPORT
1 = gives little basis for trusting or auditing the explanation
2 = weak connection to supplied evidence
3 = some evidence support but incomplete
4 = clearly grounded in supplied evidence with appropriate caution
5 = highly transparent, evidence-grounded, and appropriately calibrated

ACTIONABILITY
1 = provides no useful guidance for review
2 = limited practical value
3 = moderately useful for review
4 = clearly helps a reviewer understand what should be checked
5 = highly useful and prioritizes the most relevant evidence without overclaiming

Judge each dimension independently.
Do not reward verbosity.
Do not infer outside AML/criminal facts.

Return ONLY these fields:

CLARITY|<1-5>
TRUST_SUPPORT|<1-5>
ACTIONABILITY|<1-5>
""".strip()


def parse_quality_scores(raw):
    def extract(name):
        m = re.search(
            rf"(?i)\b{name}"
            rf"\s*\|\s*([1-5])\b",
            raw
        )

        if not m:
            raise ValueError(
                f"{name} score missing: "
                + raw[:500]
            )

        return int(
            m.group(1)
        )

    return {
        "clarity": extract(
            "CLARITY"
        ),
        "trust_support": extract(
            "TRUST_SUPPORT"
        ),
        "actionability": extract(
            "ACTIONABILITY"
        ),
    }


def judge_quality(
    e,
    txt,
    parse_retries=3
):
    evidence = json.dumps(
        e,
        ensure_ascii=False,
        indent=2,
        default=str
    )

    prompt = (
        QUALITY_JUDGE_INST
        + "\n\nSUPPLIED EVIDENCE:\n"
        + evidence
        + "\n\nEXPLANATION:\n"
        + str(txt)
    )

    last_ex = None

    for attempt in range(
        1,
        parse_retries + 1
    ):
        try:
            raw = call_model(
                LLAMA_JUDGE_MODEL,
                prompt,
                temperature=JUDGE_TEMPERATURE,
                max_tokens=QUALITY_JUDGE_MAX_TOKENS
            )

            return parse_quality_scores(
                raw
            )

        except Exception as ex:
            last_ex = ex

            if attempt < parse_retries:
                print(
                    f"⚠️ Quality judge retry "
                    f"[{attempt}/{parse_retries}] "
                    f"{type(ex).__name__}: "
                    f"{str(ex)[:220]}"
                )
                time.sleep(3)

    raise RuntimeError(
        f"Quality judge failed: {last_ex}"
    )


# ============================================================
# 8. CHECKPOINT HELPERS
# ============================================================

checkpoint_lock = threading.Lock()


def make_key(
    case_id,
    condition,
    rep
):
    return (
        str(case_id),
        str(condition),
        int(rep)
    )


def load_existing_checkpoint(
    checkpoint_path
):
    rows = []
    completed = set()

    if not os.path.exists(
        checkpoint_path
    ):
        return rows, completed

    try:
        old_df = pd.read_csv(
            checkpoint_path
        )

        if len(old_df) > 0:
            rows = old_df.to_dict(
                "records"
            )

            for _, row in old_df.iterrows():
                completed.add(
                    make_key(
                        row["case_id"],
                        row["condition"],
                        row["rep"]
                    )
                )

            print(
                f"♻️ RQ4 resume: "
                f"{len(completed)} completed"
            )

    except Exception as ex:
        print(
            "⚠️ Existing checkpoint could not be loaded:",
            ex
        )

    return rows, completed


def save_checkpoint(
    rows,
    checkpoint_path
):
    tmp_path = (
        checkpoint_path
        + ".tmp"
    )

    pd.DataFrame(
        rows
    ).to_csv(
        tmp_path,
        index=False,
        encoding="utf-8-sig"
    )

    os.replace(
        tmp_path,
        checkpoint_path
    )


# ============================================================
# 9. ONE RQ4 JOB
# ============================================================

def run_one_rq4_job(
    e,
    condition,
    rep
):
    t_all = time.time()

    t0 = time.time()

    txt = rq4_generate(
        e,
        condition
    )

    generation_sec = (
        time.time()
        - t0
    )

    t0 = time.time()

    rates = rq4_grounded_rates(
        e,
        txt
    )

    grounded_sec = (
        time.time()
        - t0
    )

    t0 = time.time()

    quality = judge_quality(
        e,
        txt
    )

    quality_sec = (
        time.time()
        - t0
    )

    total_sec = (
        time.time()
        - t_all
    )

    return {
        "case_id": str(
            e["case_id"]
        ),
        "true_label": e.get(
            "true_label"
        ),
        "prediction": e.get(
            "prediction"
        ),
        "anomaly_probability": e.get(
            "anomaly_probability"
        ),
        "condition": condition,
        "rep": int(rep),
        "explanation": txt,
        "word_count": len(
            str(txt).split()
        ),
        "generation_sec": generation_sec,
        "grounded_judge_sec": grounded_sec,
        "quality_judge_sec": quality_sec,
        "total_sec": total_sec,
        **rates,
        **quality
    }


# ============================================================
# 10. RQ4 RUN
# ============================================================

def run_rq4_only(
    evidence_list,
    reps=3,
    checkpoint_path=None,
    workers=1
):
    rows, completed = (
        load_existing_checkpoint(
            checkpoint_path
        )
    )

    jobs = []

    for e in evidence_list:
        for condition in RQ4_CONDITIONS:
            for rep in range(
                1,
                reps + 1
            ):
                key = make_key(
                    e["case_id"],
                    condition,
                    rep
                )

                if key not in completed:
                    jobs.append(
                        (
                            e,
                            condition,
                            rep,
                            key
                        )
                    )

    total_jobs = (
        len(evidence_list)
        * len(RQ4_CONDITIONS)
        * reps
    )

    done = len(
        completed
    )

    print()
    print("=" * 72)
    print("RQ4 COMPONENT ABLATION")
    print("=" * 72)
    print(
        "Cases       :",
        len(evidence_list)
    )
    print(
        "Conditions  :",
        len(RQ4_CONDITIONS)
    )
    print(
        "Repetitions :",
        reps
    )
    print(
        "Total rows  :",
        total_jobs
    )
    print(
        "Resume      :",
        done,
        "/",
        total_jobs
    )
    print(
        "Remaining   :",
        len(jobs)
    )
    print(
        "Workers     :",
        workers
    )
    print("=" * 72)

    if not jobs:
        return pd.DataFrame(
            rows
        )

    if workers <= 1:
        for e, condition, rep, key in jobs:
            row = run_one_rq4_job(
                e,
                condition,
                rep
            )

            rows.append(
                row
            )
            completed.add(
                key
            )
            done += 1

            if checkpoint_path:
                save_checkpoint(
                    rows,
                    checkpoint_path
                )

            print(
                f"✅ RQ4 {done}/{total_jobs} "
                f"| case={e['case_id']} "
                f"| {condition} "
                f"| rep={rep} "
                f"| {row['total_sec']:.1f}s"
            )

    else:
        with ThreadPoolExecutor(
            max_workers=workers
        ) as executor:

            future_map = {}

            for (
                e,
                condition,
                rep,
                key
            ) in jobs:

                fut = executor.submit(
                    run_one_rq4_job,
                    e,
                    condition,
                    rep
                )

                future_map[fut] = (
                    e,
                    condition,
                    rep,
                    key
                )

            for fut in as_completed(
                future_map
            ):
                (
                    e,
                    condition,
                    rep,
                    key
                ) = future_map[fut]

                try:
                    row = fut.result()

                except Exception as ex:
                    print(
                        f"❌ RQ4 FAILED "
                        f"| case={e['case_id']} "
                        f"| {condition} "
                        f"| rep={rep} "
                        f"| {type(ex).__name__}: "
                        f"{ex}"
                    )
                    continue

                with checkpoint_lock:
                    rows.append(
                        row
                    )
                    completed.add(
                        key
                    )
                    done += 1

                    if checkpoint_path:
                        save_checkpoint(
                            rows,
                            checkpoint_path
                        )

                print(
                    f"✅ RQ4 {done}/{total_jobs} "
                    f"| case={e['case_id']} "
                    f"| {condition} "
                    f"| rep={rep} "
                    f"| {row['total_sec']:.1f}s"
                )

    return pd.DataFrame(
        rows
    )


# ============================================================
# 11. STATISTICS HELPERS
# ============================================================

def holm_adjust(
    pvals
):
    pvals = np.asarray(
        pvals,
        dtype=float
    )

    m = len(
        pvals
    )

    if m == 0:
        return np.array([])

    order = np.argsort(
        pvals
    )

    adjusted = np.empty(
        m,
        dtype=float
    )

    running = 0.0

    for rank, idx in enumerate(
        order
    ):
        val = (
            (m - rank)
            * pvals[idx]
        )

        running = max(
            running,
            val
        )

        adjusted[idx] = min(
            1.0,
            running
        )

    return adjusted


def build_statistics(
    rq4_df,
    metrics
):
    stat_rows = []

    for metric in metrics:
        pivot = rq4_df.pivot_table(
            index=[
                "case_id",
                "rep"
            ],
            columns="condition",
            values=metric,
            aggfunc="mean"
        )

        pivot = pivot.dropna(
            subset=RQ4_CONDITIONS
        )

        if len(pivot) >= 2:
            try:
                friedman_stat, friedman_p = (
                    friedmanchisquare(
                        *[
                            pivot[c].values
                            for c
                            in RQ4_CONDITIONS
                        ]
                    )
                )

            except Exception:
                friedman_stat = np.nan
                friedman_p = np.nan

            stat_rows.append({
                "metric": metric,
                "test": "Friedman",
                "comparison": "All Conditions",
                "statistic": friedman_stat,
                "p_raw": friedman_p,
                "p_holm": np.nan,
                "n": len(pivot)
            })

        pair_results = []

        for c1, c2 in combinations(
            RQ4_CONDITIONS,
            2
        ):
            temp = pivot[
                [c1, c2]
            ].dropna()

            if len(temp) < 2:
                continue

            try:
                stat, p = wilcoxon(
                    temp[c1],
                    temp[c2]
                )

            except ValueError:
                stat = 0.0
                p = 1.0

            pair_results.append({
                "metric": metric,
                "test": "Wilcoxon",
                "comparison": (
                    f"{c1} vs {c2}"
                ),
                "statistic": stat,
                "p_raw": p,
                "n": len(temp)
            })

        if pair_results:
            adjusted = holm_adjust(
                [
                    x["p_raw"]
                    for x
                    in pair_results
                ]
            )

            for (
                result,
                adj_p
            ) in zip(
                pair_results,
                adjusted
            ):
                result["p_holm"] = (
                    adj_p
                )
                stat_rows.append(
                    result
                )

    return pd.DataFrame(
        stat_rows
    )


# ============================================================
# 12. LOAD EVIDENCE
# ============================================================

print()
print(
    "Loading evidence checkpoint..."
)

evidence_packages = (
    load_evidence_checkpoint()
)

RQ4_N_CASES = min(
    RQ4_N_CASES,
    len(evidence_packages)
)

rq4_evidence = (
    evidence_packages[
        :RQ4_N_CASES
    ]
)

print(
    "Evidence loaded:",
    len(evidence_packages)
)
print(
    "RQ4 cases:",
    len(rq4_evidence)
)


# ============================================================
# 13. EXPERIMENT PLAN
# ============================================================

estimated_rows = (
    RQ4_N_CASES
    * len(RQ4_CONDITIONS)
    * RQ4_REPS
)

estimated_qwen_calls = (
    RQ4_N_CASES
    * RQ4_REPS
    * 5
)

estimated_llama_calls = (
    estimated_rows
    * 2
)

estimated_total_calls = (
    estimated_qwen_calls
    + estimated_llama_calls
)

print()
print("=" * 72)
print("RQ4 EXPERIMENT PLAN")
print("=" * 72)
print(
    f"Cases                : "
    f"{RQ4_N_CASES}"
)
print(
    f"Conditions           : "
    f"{len(RQ4_CONDITIONS)}"
)
print(
    f"Repetitions          : "
    f"{RQ4_REPS}"
)
print(
    f"Result rows          : "
    f"{estimated_rows}"
)
print(
    f"Estimated Qwen calls : "
    f"{estimated_qwen_calls}"
)
print(
    f"Estimated Llama calls: "
    f"{estimated_llama_calls}"
)
print(
    f"Estimated API calls  : "
    f"{estimated_total_calls}"
)
print(
    f"Workers              : "
    f"{RQ4_WORKERS}"
)
print("=" * 72)


# ============================================================
# 14. RUN
# ============================================================

rq4_start = time.time()

rq4_df = run_rq4_only(
    rq4_evidence,
    reps=RQ4_REPS,
    checkpoint_path=RQ4_CHECKPOINT,
    workers=RQ4_WORKERS
)

rq4_runtime_sec = (
    time.time()
    - rq4_start
)

if rq4_df.empty:
    raise RuntimeError(
        "RQ4 result is empty."
    )


# ============================================================
# 15. COMPLETENESS CHECK
# ============================================================

expected_keys = {
    make_key(
        e["case_id"],
        condition,
        rep
    )
    for e in rq4_evidence
    for condition
    in RQ4_CONDITIONS
    for rep
    in range(
        1,
        RQ4_REPS + 1
    )
}

actual_keys = {
    make_key(
        row["case_id"],
        row["condition"],
        row["rep"]
    )
    for _, row
    in rq4_df.iterrows()
}

missing_keys = (
    expected_keys
    - actual_keys
)

print()
print(
    f"Completeness: "
    f"{len(actual_keys)}/"
    f"{len(expected_keys)}"
)

if missing_keys:
    print(
        f"⚠️ Missing jobs: "
        f"{len(missing_keys)}"
    )
else:
    print(
        "✅ All planned RQ4 jobs completed."
    )


# ============================================================
# 16. ADD METADATA / SAVE FULL RESULT
# ============================================================

rq4_df[
    "experiment_n_cases"
] = RQ4_N_CASES

rq4_df[
    "experiment_reps"
] = RQ4_REPS

rq4_df[
    "runtime_sec_total"
] = rq4_runtime_sec

rq4_df.to_csv(
    RQ4_FULL_PATH,
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 17. SUMMARY
# ============================================================

RQ4_METRICS = [
    "ESR",
    "UCR",
    "CR",
    "clarity",
    "trust_support",
    "actionability",
]

available_metrics = [
    c
    for c in RQ4_METRICS
    if c in rq4_df.columns
]

rq4_summary = (
    rq4_df
    .groupby(
        "condition"
    )[
        available_metrics
    ]
    .agg(
        ["mean", "std"]
    )
)

rq4_summary.to_csv(
    RQ4_SUMMARY_PATH,
    encoding="utf-8-sig"
)


# ============================================================
# 18. RUNTIME SUMMARY
# ============================================================

runtime_info = pd.DataFrame([
    {
        "n_cases": RQ4_N_CASES,
        "reps": RQ4_REPS,
        "conditions": len(
            RQ4_CONDITIONS
        ),
        "expected_rows": (
            estimated_rows
        ),
        "completed_rows": (
            len(actual_keys)
        ),
        "missing_rows": (
            len(missing_keys)
        ),
        "runtime_sec": (
            rq4_runtime_sec
        ),
        "runtime_min": (
            rq4_runtime_sec / 60
        ),
        "runtime_hour": (
            rq4_runtime_sec / 3600
        ),
        "mean_generation_sec": (
            pd.to_numeric(
                rq4_df[
                    "generation_sec"
                ],
                errors="coerce"
            ).mean()
        ),
        "mean_grounded_judge_sec": (
            pd.to_numeric(
                rq4_df[
                    "grounded_judge_sec"
                ],
                errors="coerce"
            ).mean()
        ),
        "mean_quality_judge_sec": (
            pd.to_numeric(
                rq4_df[
                    "quality_judge_sec"
                ],
                errors="coerce"
            ).mean()
        ),
        "mean_total_sec_per_row": (
            pd.to_numeric(
                rq4_df[
                    "total_sec"
                ],
                errors="coerce"
            ).mean()
        ),
        "estimated_qwen_calls": (
            estimated_qwen_calls
        ),
        "estimated_llama_calls": (
            estimated_llama_calls
        ),
        "estimated_total_api_calls": (
            estimated_total_calls
        ),
    }
])

runtime_info.to_csv(
    RQ4_RUNTIME_PATH,
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 19. STATISTICAL SIGNIFICANCE
# ============================================================

rq4_stats_df = (
    build_statistics(
        rq4_df,
        available_metrics
    )
)

rq4_stats_df.to_csv(
    RQ4_STATS_PATH,
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 20. PKL CHECKPOINT
# ============================================================

with open(
    RQ4_PKL_PATH,
    "wb"
) as f:
    pickle.dump(
        {
            "results_df": rq4_df,
            "summary": rq4_summary,
            "runtime_info": runtime_info,
            "stats": rq4_stats_df,
            "n_cases": RQ4_N_CASES,
            "reps": RQ4_REPS,
            "conditions": RQ4_CONDITIONS,
            "generator_model": QWEN_MODEL,
            "judge_model": LLAMA_JUDGE_MODEL,
            "expected_rows": (
                estimated_rows
            ),
            "completed_rows": (
                len(actual_keys)
            ),
            "missing_rows": (
                len(missing_keys)
            ),
        },
        f,
        protocol=pickle.HIGHEST_PROTOCOL
    )


# ============================================================
# 21. FINAL
# ============================================================

print()
print("=" * 72)
print("✅ RQ4 COMPLETE")
print("=" * 72)
print(
    "Cases       :",
    RQ4_N_CASES
)
print(
    "Conditions  :",
    len(RQ4_CONDITIONS)
)
print(
    "Reps        :",
    RQ4_REPS
)
print(
    "Expected    :",
    estimated_rows
)
print(
    "Completed   :",
    len(actual_keys)
)
print(
    "Missing     :",
    len(missing_keys)
)
print(
    "Runtime     :",
    f"{rq4_runtime_sec / 60:.2f} min"
)
print(
    "Runtime     :",
    f"{rq4_runtime_sec / 3600:.2f} hours"
)
print()
print(
    "Checkpoint  :",
    RQ4_CHECKPOINT
)
print(
    "Full result :",
    RQ4_FULL_PATH
)
print(
    "Summary     :",
    RQ4_SUMMARY_PATH
)
print(
    "Stats       :",
    RQ4_STATS_PATH
)
print(
    "Runtime     :",
    RQ4_RUNTIME_PATH
)
print(
    "PKL         :",
    RQ4_PKL_PATH
)
print("=" * 72)

try:
    display(
        rq4_summary.round(3)
    )
    display(
        rq4_stats_df
    )
    display(
        runtime_info
    )
except NameError:
    print(
        rq4_summary.round(3)
    )
    print(
        rq4_stats_df
    )
    print(
        runtime_info
    )

if missing_keys:
    print()
    print(
        "⚠️ Some tasks failed. "
        "Rerun this cell to "
        "resume pending tasks from the checkpoint."
    )
else:
    print()
    print(
        "✅ RQ4 50 cases × "
        "4 conditions × "
        "3 reps complete."
    )
